## Overview

This notebook is intended to provide some snippets of code summarizing the distribution of DERs of various types across meters of various types, including the combinations in which DERs appear together behind meters.

## Setup

In [1]:
import getpass
import urllib.parse
import pandas as pd

**Connection**

Enter the EDM server address and the login credentials provided by Awesense. If you do not have the credentials, or have any trouble connecting, please contact api@awesense.com.
<span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [2]:
edm_address = getpass.getpass(prompt='EDM server address: ')

print('\nEDM login information')
edm_name = getpass.getpass(prompt='Username: ')
edm_password = getpass.getpass(prompt='Password: ')
edm_password = urllib.parse.quote(edm_password)

get_ipython().run_line_magic('load_ext', 'sql')
get_ipython().run_line_magic('sql', 'postgresql://$edm_name:$edm_password@$edm_address/edm')
get_ipython().run_line_magic('config', 'SqlMagic.displaycon = False')
get_ipython().run_line_magic('config', 'SqlMagic.feedback = False')

# Delete the credential variables for security purposes.

del edm_name, edm_password

EDM server address: ········

EDM login information
Username: ········
Password: ········


## DER combinations behind meters

**Input Parameters**

Enter the grid ID of interest.

In [3]:
grid_id = input('Enter grid ID: ') # awefice

Enter grid ID: awefice


**Retrieve all meter-DER pairs where the DER is downstream of (behind) the meter.**

In [4]:
%%sql meter_ders << SELECT ge1.grid_element_id as meter_id,
    ge1.meta ->> 'type_of_consumer' as meter_type,
    ge2.type as der_type,
    ge2.grid_element_id as der_id
FROM grid_element ge1
JOIN grid_get_downstream('awefice', ge1.grid_element_id, False) ge2 ON True
WHERE ge1.grid_id = '{grid_id}' AND ge1.type = 'Meter'
    AND (ge2.type='EVCharger' OR ge2.type='Photovoltaic' OR ge2.type='Battery')
ORDER BY meter_type, meter_id;


Returning data to local variable meter_ders


In [5]:
# Convert the results to a data frame. 
df_meter_ders = meter_ders.DataFrame()
df_meter_ders

,meter_id,meter_type,der_type,der_id
0,m_10,business,EVCharger,evch_10
1,m_10,business,Photovoltaic,pv_10
2,m_10,business,Battery,batt_10
3,m_12,business,EVCharger,evch_12
4,m_24,business,Photovoltaic,pv_24
5,m_24,business,Battery,batt_24
6,m_24,business,EVCharger,evch_24
7,m_11,industrial,Photovoltaic,pv_11
8,m_19,residential,Battery,batt_19
9,m_19,residential,Photovoltaic,pv_19


**Find the DER combinations**

In [6]:
# Pivotal the table by DER type
df_matrix = df_meter_ders.pivot(index=['meter_type', 'meter_id'], columns='der_type', values='der_id')
df_matrix

der_type              Battery EVCharger Photovoltaic
meter_type  meter_id                                
business    m_10      batt_10   evch_10        pv_10
            m_12          NaN   evch_12          NaN
            m_24      batt_24   evch_24        pv_24
industrial  m_11          NaN       NaN        pv_11
residential m_19      batt_19       NaN        pv_19
            m_20          NaN   evch_20          NaN
            m_27          NaN       NaN        pv_27
            m_3           NaN    evch_3          NaN
            m_4        batt_4       NaN         pv_4

In [7]:
# Label the DER combinations
df_matrix['DERs Present'] = df_matrix['Battery'].isnull().map({True: '_', False: 'B'}) \
    + df_matrix['EVCharger'].isnull().map({True: '_', False: 'E'}) \
    + df_matrix['Photovoltaic'].isnull().map({True: '_', False: 'P'})
df_matrix

der_type              Battery EVCharger Photovoltaic DERs Present
meter_type  meter_id                                             
business    m_10      batt_10   evch_10        pv_10          BEP
            m_12          NaN   evch_12          NaN          _E_
            m_24      batt_24   evch_24        pv_24          BEP
industrial  m_11          NaN       NaN        pv_11          __P
residential m_19      batt_19       NaN        pv_19          B_P
            m_20          NaN   evch_20          NaN          _E_
            m_27          NaN       NaN        pv_27          __P
            m_3           NaN    evch_3          NaN          _E_
            m_4        batt_4       NaN         pv_4          B_P

In [8]:
# Pivot again to list and count the meters for each DER combination, grouped by meter type.
df_matrix.rename_axis(None, axis=1).reset_index().pivot_table(index=['meter_type'], columns='DERs Present', values='meter_id', aggfunc=lambda x: str(len(x)) + ': ' + ', '.join(x.astype(str)))

DERs Present,BEP,B_P,_E_,__P
meter_type,,,,
business,"2: m_10, m_24",NaN,1: m_12,NaN
industrial,NaN,NaN,NaN,1: m_11
residential,NaN,"2: m_19, m_4","2: m_20, m_3",1: m_27
